## Install Packages and Set Up Path

In [2]:
# Use the project `requirements.txt` located one level up..
import sys, os
print('Installing requirements from ../requirements.txt (this may take a few minutes)')
# Preferred in-notebook installer
get_ipython()
%pip install -r ../requirements.txt
# Set up PYTHONPATH to include infrapy path
local_infrapy_path = r'C:\Users\ISLA-ENG-01\Documents\Projects\infrapy-ISLA'
if local_infrapy_path not in sys.path:
    sys.path.insert(0, local_infrapy_path)
    os.environ['PYTHONPATH'] = local_infrapy_path
print(f'PYTHONPATH set to {local_infrapy_path} \n Environment is now ready')

Installing requirements from ../requirements.txt (this may take a few minutes)
Note: you may need to restart the kernel to use updated packages.
PYTHONPATH set to C:\Users\ISLA-ENG-01\Documents\Projects\infrapy-ISLA 
 Environment is now ready


## Load in Variables from Configuration File


In [10]:
import configparser

# InfraPy imports (use functions directly from the installed/source package)
from infrapy.utils import config as infraconfig

user_config = configparser.ConfigParser()
user_config.read('../config/base_test.ini')

# Helper to pull params from InfraPy user config
def cfg(param_section, param_name, dtype='float', cli_val=None):
    return infraconfig.set_param(user_config, param_section, param_name, cli_val, dtype)

# beamforming parameters
freq_min = cfg('FK', 'freq_min', 'float') 
freq_max = cfg('FK', 'freq_max', 'float') 
back_az_min = cfg('FK', 'back_az_min', 'float') 
back_az_max = cfg('FK', 'back_az_max', 'float') 
back_az_step = cfg('FK', 'back_az_step', 'float') 
trace_vel_min = cfg('FK', 'trace_vel_min', 'float') 
trace_vel_max = cfg('FK', 'trace_vel_max', 'float') 
trace_vel_step = cfg('FK', 'trace_vel_step', 'float') 
method = infraconfig.set_param(user_config, 'FK', 'method', None, 'string') 
signal_start = infraconfig.set_param(user_config, 'FK', 'signal_start', None, 'string')
signal_end = infraconfig.set_param(user_config, 'FK', 'signal_end', None, 'string')
noise_start = infraconfig.set_param(user_config, 'FK', 'noise_start', None, 'string')
noise_end = infraconfig.set_param(user_config, 'FK', 'noise_end', None, 'string')
window_len = cfg('FK', 'window_len', 'float') 
sub_window_len = cfg('FK', 'sub_window_len', 'float')
window_step = cfg('FK', 'window_step', 'float') 
cpu_cnt = infraconfig.set_param(user_config, 'FK', 'cpu_cnt', None, 'int')

# detection parameters
fd_window_len = cfg('FD', 'window_len', 'float') 
p_value = cfg('FD', 'p_value', 'float') 
min_duration = cfg('FD', 'min_duration', 'float') 
back_az_width = cfg('FD', 'back_az_width', 'float') 
fixed_thresh = cfg('FD', 'fixed_thresh', 'float')
thresh_ceil = cfg('FD', 'thresh_ceil', 'float')
return_thresh = infraconfig.set_param(user_config, 'FD', 'return_thresh', None, 'bool') or False
# NOTE: Merge detections currently needs to be improved. Should investigate how they associate nearby detections (time window, etc).
merge_dets = infraconfig.set_param(user_config, 'FD', 'merge_dets', None, 'bool') or True

### Optional: use MCML/MVDR (Capon) beamforming
Toggle this if you want the automated detector to run the adaptive Capon/MCML beamformer instead of the default Bartlett-style FK.

In [ ]:
# Optionally override the beamforming method to MCML/MVDR (Capon)
use_mcml_mvdr = True
if use_mcml_mvdr:
    method = "capon"
    print("Beamforming method set to MCML/MVDR (Capon)")
else:
    print(f"Beamforming method left as configured: {method}")

## Run Beamforming and Detection Algorithm on Seedlink

In [ ]:
"""
Questions:
1) What attributes should we be pulling out? Currently
"""


import obspy
from obspy.core.util import AttribDict
from obspy.clients.fdsn import Client
from obspy.clients.seedlink import Client as Client_seedlink
import numpy as np
import time
import json
import os
# InfraPy imports
from infrapy.utils import data_io
from infrapy.detection import beamforming_new as fkd

wf_client = 1 # Flag to pull data from IRIS (0) or seedlink (1)
real_time = 1 # Flag to for static time frame (0) or real-time processing (1)
i = 0
# Currently using while loop for simplicity, i < 71 corresponds with ~24 hours w/ currently set up at 10 minute intervals with 1.5 minutes of overlap.
# For the current setup it takes about 2.25 minutes to run 10 minutes at .5-8 Hz range.
while i < 600:  
    stop_watch = time.time()
    # Adding event params
    EVENT_CONFIG = {
        'name': 'auto_infrapy_test',
        'network': 'IM',
        'station': 'I59*',
        'location': '',
        'channel': 'BDF',
        'start_time': obspy.UTCDateTime()-720 if(real_time) else obspy.UTCDateTime('2025-12-10T18:30:00.000000Z'),
        'end_time': obspy.UTCDateTime()-120 if(real_time) else obspy.UTCDateTime('2025-12-10T19:00:00.000000Z'),
    }

    # Set parameters from the event config
    name = EVENT_CONFIG['name']
    network = EVENT_CONFIG['network']
    station = EVENT_CONFIG['station']
    location = EVENT_CONFIG['location']
    channel = EVENT_CONFIG['channel']
    if (not i):
        t1 = EVENT_CONFIG['start_time']-480
    else:
        t1 = EVENT_CONFIG['start_time']
    t2 = EVENT_CONFIG['end_time']

    # Get waveforms from IRIS or seedlink
    if i == 0:
        try:
            client = Client('IRIS')
            inventory = client.get_stations(network=network,
                                            station=station,
                                            location=location,
                                            channel=channel,
                                            starttime=t1,
                                            endtime=t2,
                                            level="response")


        except:
            print('Error fetching data from FDSN client. Please check network/station codes and time range.')
            break

    # Set up seedlink and take in stream
    LOCAL_SEEDLINK = "192.168.112.200"

    seed = Client_seedlink(LOCAL_SEEDLINK, port=18000, timeout=180)
    try:
        if(wf_client):
            g_stream = seed.get_waveforms(network=network, location=location, station=station,
                                    channel=channel, starttime=t1, endtime=t2)
            if len(g_stream) > 0:
                print("Data Found on seedlink")
            else:
                print("Error fetching data from Seedlink. WiFi is correct, possibly an issue with retrieving data from CTBTO.")
        else:
            g_stream = client.get_waveforms(network=network, location=location, station=station,
                                    channel=channel, starttime=t1, endtime=t2)
            if len(g_stream) > 0:
                print("Data Found on IRIS")
    except:
        print("Error fetching data")
    # Add coordinates to stream using inv feteched from IRIS
    latlon = []
    for tr in g_stream:
        coords = inventory.get_coordinates(f"{network}.{tr.stats.station}.{location}.{channel}", t1)
        tr.stats.coordinates = AttribDict({
        'latitude': coords['latitude'],
        'elevation': coords['elevation'],
        'longitude': coords['longitude']})
        latlon.append((coords['latitude'], coords['longitude']))
        print(tr.stats.starttime, tr.stats.station)
    print(f'Fetched {len(g_stream)} traces from {network}.{station}.')

    # Get the centroid of the array. Standard coords are fine bc array isn't big enough for geodeisic shifting to occur.
    centroid = np.mean([lat for lat, lon in latlon]), np.mean([lon for lat, lon in latlon])
    array_lat, array_lon = centroid

    strm = g_stream.copy()
    print(f'Run iteration {i}')

    # Noise is calculated based on the previous stream. If the previous stream has detections it wil use the most current stream that does not have any detections.
    if (not i):
        # i==0 is a special case in which the previous 8 minutes of signal as the baseline noise.
        prev_start_time = t1
        dets = 0
        noise_start = t1
        noise_end = t1 + 480
        strm = strm.trim(t1+480, t2)
        n_strm = g_stream.trim(t1, t1+480)
    else:
        if(not dets):
            noise_start = prev_start_time
            noise_end = prev_start_time + 480
        else:
            pass
    
    print(f"  Noise window: {noise_start} to {noise_end}")
    # Compute noise and signal indices in seconds relative to stream start (t1).
    noise_len = (noise_end - noise_start)
    subset_start = t1 if(i) else t1 + 480 
    str_name = name + "_" + t1.strftime("%Y%m%d_%H%M%S")
    print(f"Running Detection {subset_start} to {t2}")
    print(f"Processing stream: {str_name}")


    # Setup bf inputs based on config file
    back_az_vals = np.arange(back_az_min, back_az_max, back_az_step)
    trc_vel_vals = np.arange(trace_vel_min, trace_vel_max, trace_vel_step)
    
    # Run beamforming
    print(f"Running {method} beamforming")

    x, t, t0, geom = fkd.stream_to_array_data(strm, latlon=latlon)
    M, N = x.shape

    slowness = fkd.build_slowness(back_az_vals, trc_vel_vals)
    delays = fkd.compute_delays(geom, slowness)

    # Beamforming returns beam_power as a 3D array. Need to look into what actually is returned and best way to access this data
    beam_times, beam_peaks, beam_power = fkd.auto_run_bf(
                                                    (subset_start - t1),
                                                    (t2 - subset_start), freq_band=[freq_min, freq_max],
                                                    window_len=window_len,
                                                    sub_window_len=sub_window_len,
                                                    window_step=window_step,
                                                    method=method,
                                                    back_az_vals=back_az_vals,
                                                    trc_vel_vals=trc_vel_vals,
                                                    array_data=[x, t, t0, geom],
                                                    delays=delays
                                                    )

    # Save beam_power and beam_times for plotting
    rd_out_temp = "../results/" + t1.strftime("%Y/%m/%d/") + str_name + '_raw_data.txt'
    np.save(rd_out_temp.replace('_raw_data.txt', '_beam_power.npy'), beam_power)
    np.save(rd_out_temp.replace('_raw_data.txt', '_beam_times.npy'), beam_times)

    # Run detection


    print(f"Running FD detection")
    # Compute noise _fstat for detection auto threshold ; IPBeamformingWidget.py lines 1402 -> 1449 
    TB_prod = (freq_max - freq_min) * window_len
    if(fixed_thresh):
        thresh = fixed_thresh
    else:
        # 
        # If detections were found thresh will be the same as previous valid fstat threshold. If not recompute with the previous timeslot
        if (not i):
            n_x, n_t, n_t0, n_geom = fkd.stream_to_array_data(n_strm, latlon=latlon)
            n_delays = fkd.compute_delays(n_geom, slowness)
            thresh = fkd.adjust_thresh_noise([n_x, n_t, n_t0, n_geom], window_len, sub_window_len, noise_len, window_step, freq_min, freq_max, method, back_az_vals, trc_vel_vals, n_delays, p_value, TB_prod)
        elif (dets):
            thresh = prev_thresh
            print(f"Threshold Used: {thresh} Check to see if this matches the previous f-stat threshold (It should)")
        else:
            thresh = new_thresh
            
    min_seq = int(max(2, min_duration / (window_step)))
    det_results = fkd.run_fd(beam_times, beam_peaks, window_len, TB_prod, len(strm), 
                      p_value, min_seq, back_az_width, thresh, thresh_ceil, 
                      return_thresh, merge_dets)
    dets = det_results[0] if return_thresh else det_results
    
    det_list = []
    for det_info in dets:
        det = data_io.define_detection(det_info, [array_lat, array_lon], len(strm), 
                                      [freq_min, freq_max], note="Automated run", method=method)
        det_list.append(det)
    

    print(f"Detection Complete")

# Save detections
    if len(det_list) > 0:
        det_fpath = "../results/" + t1.strftime("%Y/%m/%d/")
        try:
            if not os.path.isdir(det_fpath):
                print("Making New Folder")
                os.makedirs(det_fpath, exist_ok=True)
            else:
                pass
        except:
            print("Error making folder, please investigate issues")
            det_fpath = "../results/bin/"


        det_out = det_fpath + str_name + '_detections.json'
        dets = 1
        prev_thresh = thresh
        print(f"  Found {len(det_list)} detections, writing to {det_out}\nNew Threshold: {prev_thresh}")
        str_info = [strm[0].stats.network,
                       strm[0].stats.station + "-" + str(len(strm[0])),
                       strm[0].stats.channel]
        data_io.detection_list_to_json(det_out, det_list, str_info)
        
        with open(det_out, 'r') as f:
            dets_data = json.load(f)
        
        for det in dets_data:
            det['Latitude'] = array_lat
            det['Longitude'] = array_lon
            det['Noise'] = f"{noise_start} to {noise_end}"
        
        with open(det_out, 'w') as f:
            json.dump(dets_data, f, indent=4)

        # If there is a detection save off all raw data
        dt = np.array([(tn - np.datetime64(strm[0].stats.starttime)).astype('m8[ms]').astype(float) * 1.0e-3 for tn in beam_times])
        raw_data = np.hstack((np.atleast_2d(dt).T, beam_peaks))
        rd_header = data_io.fk_header(strm, latlon, freq_min, freq_max, back_az_min, back_az_max, 
                                  back_az_step, trace_vel_min, trace_vel_max, trace_vel_step, 
                                  method, subset_start, t2, noise_start, noise_end, 
                                  window_len, sub_window_len, window_step)
    
        rd_out = det_fpath + str_name + '_raw_data.txt'
        np.savetxt(rd_out, raw_data, header=rd_header)
        print(f"  Wrote FK results to {rd_out}")
    else:
        dets = 0
        prev_start_time = subset_start
        new_thresh = fkd.adjust_thresh_noise([x, t, t0, geom], window_len, sub_window_len, noise_len, window_step, freq_min, freq_max, method, back_az_vals, trc_vel_vals, delays, p_value, TB_prod)
        print(f"No detections found.\nNew Threshold: {new_thresh}")
    if(real_time):
        T = time.time() - stop_watch
        print(f"Sleeping for {510-T} seconds until {obspy.UTCDateTime()+(510-T)-36000} (HST)")
        time.sleep(510-T)
    i += 1

Error fetching data


NameError: name 'g_stream' is not defined

In [5]:
    # Save beam_power and beam_times for plotting
rd_out_temp = "../results/" + t1.strftime("%Y/%m/%d/") + str_name + '_raw_data.txt'
os.makedirs(os.path.dirname(rd_out_temp), exist_ok=True)
np.save(rd_out_temp.replace('_raw_data.txt', '_beam_power.npy'), beam_power)
np.save(rd_out_temp.replace('_raw_data.txt', '_beam_times.npy'), beam_times)

NameError: name 't1' is not defined

In [1]:
import matplotlib.pyplot as plt
import glob
from scipy.interpolate import griddata
import os
import json
import numpy as np

# InfraPy imports
from infrapy.utils import data_io
from infrapy.detection import beamforming_new as fkd

# Ensure slowness/grid parameters are available (fallback to config file if not yet loaded)
def ensure_slowness_params():
    global back_az_min, back_az_max, back_az_step, trace_vel_min, trace_vel_max, trace_vel_step
    global back_az_vals, trc_vel_vals
    needed = [
        'back_az_min', 'back_az_max', 'back_az_step',
        'trace_vel_min', 'trace_vel_max', 'trace_vel_step'
    ]
    missing = [n for n in needed if n not in globals()]
    if missing:
        import configparser
        from infrapy.utils import config as infraconfig
        cfg = configparser.ConfigParser()
        cfg.read('../config/base_test.ini')
        def pull(section, key, dtype='float'):
            return infraconfig.set_param(cfg, section, key, None, dtype)
        back_az_min = pull('FK', 'back_az_min', 'float')
        back_az_max = pull('FK', 'back_az_max', 'float')
        back_az_step = pull('FK', 'back_az_step', 'float')
        trace_vel_min = pull('FK', 'trace_vel_min', 'float')
        trace_vel_max = pull('FK', 'trace_vel_max', 'float')
        trace_vel_step = pull('FK', 'trace_vel_step', 'float')
    back_az_vals = np.arange(back_az_min, back_az_max, back_az_step)
    trc_vel_vals = np.arange(trace_vel_min, trace_vel_max, trace_vel_step)

# Helper to load beam data from raw txt (fallback when npy files are missing)
def load_beam_from_txt(raw_txt_path):
    t0_str = None
    with open(raw_txt_path, 'r') as f:
        for line in f:
            if not line.startswith('#'):
                break
            if '  t0:' in line:
                t0_str = line.split('t0:')[1].strip()
    if t0_str is None:
        raise ValueError(f"t0 not found in header for {raw_txt_path}")

    data = np.loadtxt(raw_txt_path)
    times_rel = data[:, 0]
    beam_matrix = data[:, 1:]
    t0 = np.datetime64(t0_str)
    return t0, times_rel, beam_matrix

# Find all detection files
det_files = glob.glob('../results/**/*.json', recursive=True)

all_dets = []
for f in det_files:
    with open(f, 'r') as file:
        data = json.load(file)
        for det in data:
            all_dets.append((det, f))

# Ensure grid params ready
ensure_slowness_params()

# Process each detection
for det, det_file in all_dets:
    print(f"Processing detection at {det['Time (UTC)']} with F-Stat {det['F Stat.']}")

    # Paths for data
    beam_power_file = det_file.replace('_detections.json', '_beam_power.npy')
    beam_times_file = det_file.replace('_detections.json', '_beam_times.npy')
    raw_txt_file = det_file.replace('_detections.json', '_raw_data.txt')

    beam_slice = None
    det_time = np.datetime64(det['Time (UTC)'])

    if os.path.exists(beam_power_file) and os.path.exists(beam_times_file):
        beam_power = np.load(beam_power_file)
        beam_times = np.load(beam_times_file)
        idx = np.argmin(np.abs(beam_times - det_time))
        beam_slice = np.mean(beam_power[idx], axis=1)  # average over channels
    elif os.path.exists(raw_txt_file):
        t0, times_rel, beam_matrix = load_beam_from_txt(raw_txt_file)
        det_rel_sec = (det_time - t0) / np.timedelta64(1, 's')
        idx = np.argmin(np.abs(times_rel - det_rel_sec))
        beam_slice = beam_matrix[idx]
    else:
        print(f"No beam data found for {det_file}")
        continue

    # Build slowness
    slowness = fkd.build_slowness(back_az_vals, trc_vel_vals)
    slow_x = slowness[:, 0]
    slow_y = slowness[:, 1]

    # Create grid
    slow_x_min = np.min(slow_x)
    slow_x_max = np.max(slow_x)
    slow_y_min = np.min(slow_y)
    slow_y_max = np.max(slow_y)
    grid_x, grid_y = np.mgrid[slow_x_min:slow_x_max:300j, slow_y_min:slow_y_max:300j]

    points = np.column_stack((slow_x, slow_y))
    grid_z = griddata(points, beam_slice, (grid_x, grid_y), method='linear')

    # Plot
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(grid_z.T, extent=[slow_x_min, slow_x_max, slow_y_min, slow_y_max], origin='lower', cmap='jet', aspect='equal')

    # Add circles (outer and inner)
    circle_outer = plt.Circle((0, 0), 1/trace_vel_min, fill=False, color='k', linewidth=3)
    ax.add_artist(circle_outer)
    circle_inner = plt.Circle((0, 0), 1/trace_vel_max, fill=False, color='k', linewidth=3)
    ax.add_artist(circle_inner)

    # Add radials every 45 degrees
    for az in range(0, 360, 45):
        x = (1/trace_vel_min) * np.sin(np.radians(az))
        y = (1/trace_vel_min) * np.cos(np.radians(az))
        ax.plot([0, x], [0, y], 'k:', linewidth=1)

    # Remove ticks and labels to match InfraView style
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')

    # Title for identification
    ax.set_title(f"Detection at {det['Time (UTC)']} (F={det['F Stat.']:.2f})")

    # Colorbar
    plt.colorbar(im, ax=ax, label='Beam Power')

    # Save the plot
    safe_time = det['Time (UTC)'].replace(':', '-').replace(' ', '_')
    plot_filename = det_file.replace('_detections.json', f'_polar_{safe_time}_{det["F Stat."]:.2f}.png')
    plt.savefig(plot_filename)
    plt.close(fig)
    print(f"Saved plot to {plot_filename}")

Processing detection at 2025-12-11T02:02:49.750000 with F-Stat 2.5552


C:\Users\ISLA-ENG-01\AppData\Local\Temp\ipykernel_20212\1944138413.py:52: DeprecationWarning: parsing timezone aware datetimes is deprecated; this will raise an error in the future
  t0 = np.datetime64(t0_str)


ValueError: different number of values and points